<h1 dir='rtl'>Optical flow</h1>

<h4 dir='rtl'>Optical flow الگوی حرکت ظاهری اجسام بین دو فریم متوالی است که در اثر حرکت جسم یا دوربین ایجاد میشود. درواقع با کمک این تکنیک جهت و سرعت اجسام داخل تصویر رو متوجه میشویم.</h4>

## Lucas-Kanda Optimal flow in OpenCV

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [2]:
cap= cv2.VideoCapture('slow_traffic_small.mp4')

# params for shiTomasi corner detection
feature_params= dict(maxCorners= 100, qualityLevel= 0.3, minDistance= 7, blockSize= 7)

lk_params= dict(winSize= (15,15), maxLevel= 2, criteria= (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

color= np.random.randint(0, 255, (100,3))

_, old_frame= cap.read()
old_gray= cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)

p0= cv2.goodFeaturesToTrack(old_gray, mask= None, **feature_params)
mask= np.zeros_like(old_frame)

while True:
    ret, frame= cap.read()

    if not ret:
        print("frame Error :(")
        break

    frame_gray= cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    p1, st, err= cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)

    if p1 is not None:
        good_new= p1[st==1]
        good_old= p0[st==1]
    for i, (new,old) in enumerate(zip(good_new, good_old)):
        a,b= new.ravel()
        c,d= old.ravel()
        mask= cv2.line(mask, (int(a), int(b)), (int(c), int(d)), color[i].tolist(), 2)
        frame= cv2.circle(frame, (int(a), int(b)), 5, color[i].tolist(), -1)

    img= cv2.add(frame, mask)
    cv2.imshow('frame', img)

    k= cv2.waitKey(30)&0xff

    if k == 27:
        break

    old_gray= frame_gray.copy()
    p0= good_new.reshape(-1,1,2)
cv2.destroyAllWindows()

## Dense Optical Flow in OpenCV

In [4]:
cap= cv2.VideoCapture('vtest.avi')

ret,frame1= cap.read()

prvs= cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
hsv= np.zeros_like(frame1)
hsv[...,1]= 255

while True:
    ret, frame2= cap.read()

    if not ret:
        print("frame Error ;(")
        break

    next= cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)
    flow= cv2.calcOpticalFlowFarneback(prvs, next, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    mag, ang= cv2.cartToPolar(flow[...,0], flow[...,1])
    hsv[...,0]= ang*180/np.pi/2
    hsv[...,2]= cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
    bgr= cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    cv2.imshow('frame2', bgr)

    k= cv2.waitKey(30)&0xff

    if k==27:
        break

    elif k==ord('s'):
        cv2.imwrite('Opticalfb.png', frame2)
        cv2.imwrite("opricalhsv.png", bgr)

    prvs= next
cv2.destroyAllWindows()

In [5]:
def draw_flow(img, flow, step= 16):

    h,w= img.shape[:2]
    y,x= np.mgrid[step/2:h:step, step/2:w:step].reshape(2,-1).astype(int)
    fx, fy= flow[y,x].T

    lines= np.vstack([x, y, x-fx, y-fy]).T.reshape(-1,2,2)
    lines= np.int32(lines+0.5)

    img_bgr= cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    cv2.polylines(img_bgr, lines, 0, (0, 255,0))

    for (x1,y1), (_x2, _y2) in lines:
        cv2.circle(img_bgr, (x1,y1), 1, (0,255,0), -1)

    return img_bgr

def draw_hsv(flow):

    h,w= flow.shape[:2]
    fx, fy= flow[:,:,0], flow[:,:,1]
    
    ang = np.arctan2(fy, fx) + np.pi
    v = np.sqrt(fx*fx+fy*fy)

    hsv = np.zeros((h, w, 3), np.uint8)
    hsv[...,0] = ang*(180/np.pi/2)
    hsv[...,1] = 255
    hsv[...,2] = np.minimum(v*4, 255)
    bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    return bgr

cap = cv2.VideoCapture(0)

suc, prev = cap.read()
prevgray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)

while True:

    suc, img = cap.read()
    img= cv2.flip(img, 1)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    flow = cv2.calcOpticalFlowFarneback(prevgray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    
    prevgray = gray

    cv2.imshow('flow', draw_flow(gray, flow))
    cv2.imshow('flow HSV', draw_hsv(flow))

    key = cv2.waitKey(5)
    if key == 27:
        break

cap.release()
cv2.destroyAllWindows()
        